# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mahmoud-Beram/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My Rule (Snippet Fix): A page is worth reviewing if it has high visibility (over 1,000 impressions) and ranks on Page 1 (average position <= 10), but completely fails to attract clicks (CTR < 1%).

Reason Code: CTR_TOO_LOW_FOR_PAGE_1 Action: SNIPPET_FIX

Signal 1 Verdict: CONFIRMED. The bucket table below proves that Page 1 naturally commands a high CTR. If an article violates this trend, it's an anomaly worth fixing.

Signal 2 Verdict: CONFIRMED. The second bucket table proves that high-impression articles drive the bulk of potential traffic, making them the most valuable to fix.

In [1]:
from huggingface_hub import hf_hub_download
from google.colab import userdata
import duckdb


hf_token = userdata.get('HF_TOKEN').strip()
print("Downloading file from Hugging Face... please wait.")
local_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=hf_token
)
print("Download complete! Running Signal 1...")

# 2. Connection
con = duckdb.connect()

# 3. Signal 1 Test: Position vs CTR
print("\n--- Signal 1: CTR vs Position Page ---")
signal1_df = con.execute(f"""
    WITH article_stats AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) as total_imp,
            SUM(gsc_clicks) as total_clicks,
            AVG(gsc_avg_position) as avg_pos
        FROM '{local_file}'
        GROUP BY content_hash_id
        HAVING total_imp > 0
    )
    SELECT
        CASE
            WHEN avg_pos <= 10 THEN 'Page 1 (Pos 1-10)'
            WHEN avg_pos <= 20 THEN 'Page 2 (Pos 11-20)'
            WHEN avg_pos <= 30 THEN 'Page 3 (Pos 21-30)'
            ELSE 'Buried (Page 4+)'
        END AS position_bucket,

        COUNT(content_hash_id) AS n_articles,
        AVG(total_clicks / total_imp) * 100 AS avg_ctr_percentage

    FROM article_stats
    GROUP BY position_bucket
    ORDER BY avg_ctr_percentage DESC
""").df()

display(signal1_df)


Download complete! Running Signal 1...

--- Signal 1: CTR vs Position Page ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_bucket,n_articles,avg_ctr_percentage
0,Page 1 (Pos 1-10),99566,0.624545
1,Page 2 (Pos 11-20),32203,0.321119
2,Page 3 (Pos 21-30),17173,0.254824
3,Buried (Page 4+),27796,0.154422


In [2]:
# 4. Signal 2 Test: Impressions Volume vs Total Clicks
print("\n--- Signal 2: Impressions Volume vs Total Clicks ---")
signal2_df = con.execute(f"""
    WITH article_stats AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) as total_imp,
            SUM(gsc_clicks) as total_clicks
        FROM '{local_file}'
        GROUP BY content_hash_id
        HAVING total_imp > 0
    )
    SELECT
        CASE
            WHEN total_imp < 100 THEN '1. Low (< 100)'
            WHEN total_imp < 1000 THEN '2. Medium (100 - 1K)'
            WHEN total_imp < 10000 THEN '3. High (1K - 10K)'
            ELSE '4. Viral (> 10K)'
        END AS volume_bucket,

        COUNT(content_hash_id) AS n_articles,
        SUM(total_clicks) AS bucket_total_clicks

    FROM article_stats
    GROUP BY volume_bucket
    ORDER BY volume_bucket ASC
""").df()

total_site_clicks = signal2_df['bucket_total_clicks'].sum()
signal2_df['clicks_percentage_%'] = (signal2_df['bucket_total_clicks'] / total_site_clicks) * 100

display(signal2_df)



--- Signal 2: Impressions Volume vs Total Clicks ---


,volume_bucket,n_articles,bucket_total_clicks,clicks_percentage_%
0,1. Low (< 100),75297,6457.0,0.785684
1,2. Medium (100 - 1K),56383,53028.0,6.452414
2,3. High (1K - 10K),39181,384511.0,46.787056
3,4. Viral (> 10K),5877,377836.0,45.974846


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
# Section 2: Building the Ranked Queue
print("\n--- Section 2: Building the Ranked Queue ---")
rule_df = con.execute(f"""
    WITH article_stats AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) as total_imp,
            SUM(gsc_clicks) as total_clicks,
            AVG(gsc_avg_position) as avg_pos
        FROM '{local_file}'
        GROUP BY content_hash_id
        HAVING total_imp > 0
    )
    SELECT
        content_hash_id,
        total_imp AS score,
        'SNIPPET_FIX' AS action,
        'CTR_TOO_LOW_FOR_PAGE_1' AS reason_code,
        total_imp,
        total_clicks,
        avg_pos,
        (total_clicks / total_imp) AS ctr
    FROM article_stats
    WHERE
        total_imp > 1000
        AND avg_pos <= 10
        AND (total_clicks / total_imp) < 0.01
    ORDER BY score DESC
""").df()

import os
os.makedirs('work/outputs', exist_ok=True)
rule_df.to_csv('work/outputs/baseline_action_score.csv', index=False)

print(f"Done! Found {len(rule_df)} articles that need a Snippet Fix.")
display(rule_df[['content_hash_id', 'score', 'action', 'reason_code', 'ctr']].head(10))



--- Section 2: Building the Ranked Queue ---
Done! Found 28183 articles that need a Snippet Fix.


,content_hash_id,score,action,reason_code,ctr
0,content_eadb33b5df496f4a,617124.0,SNIPPET_FIX,CTR_TOO_LOW_FOR_PAGE_1,0.009185
1,content_ec2e0346994fb5a5,245276.0,SNIPPET_FIX,CTR_TOO_LOW_FOR_PAGE_1,0.006034
2,content_0e03de7680314cd5,221310.0,SNIPPET_FIX,CTR_TOO_LOW_FOR_PAGE_1,0.003253
3,content_44f34c0a90047651,212404.0,SNIPPET_FIX,CTR_TOO_LOW_FOR_PAGE_1,0.000113
4,content_7172a7fad43f0998,205867.0,SNIPPET_FIX,CTR_TOO_LOW_FOR_PAGE_1,0.004187
5,content_8d7d99f109e19aa2,203497.0,SNIPPET_FIX,CTR_TOO_LOW_FOR_PAGE_1,0.001420
6,content_f107e54b10b43725,195997.0,SNIPPET_FIX,CTR_TOO_LOW_FOR_PAGE_1,0.005082
7,content_b99ea6861864dea5,194337.0,SNIPPET_FIX,CTR_TOO_LOW_FOR_PAGE_1,0.001858
8,content_4ffe18112a5642e3,186983.0,SNIPPET_FIX,CTR_TOO_LOW_FOR_PAGE_1,0.003134
9,content_acbcc847f8996314,170808.0,SNIPPET_FIX,CTR_TOO_LOW_FOR_PAGE_1,0.001534


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top 10 Review (Snippet Fixes): All top 10 articles share the same mechanical issue: High Impressions (>170k), Page 1 Position, but extremely poor CTR (<1%).

1. content_eadb33... | Action: SNIPPET_FIX | Reason: CTR_TOO_LOW_FOR_PAGE_1 | What makes it wrong: If the search intent is answered directly on Google (Zero-click search). 2. content_ec2e03... | Action: SNIPPET_FIX | Reason: CTR_TOO_LOW_FOR_PAGE_1 | What makes it wrong: If the keyword is a competitor's brand name (people won't click us). 3. content_0e03de... | Action: SNIPPET_FIX | Reason: CTR_TOO_LOW_FOR_PAGE_1 | What makes it wrong: If the page is ranking for a high-volume but irrelevant keyword. 4. content_44f34c... | Action: SNIPPET_FIX | Reason: CTR_TOO_LOW_FOR_PAGE_1 | What makes it wrong: (Same zero-click risk as #1). 5. content_7172a7... | Action: SNIPPET_FIX | Reason: CTR_TOO_LOW_FOR_PAGE_1 | What makes it wrong: If our meta title is actually good, but Google is rewriting it with something bad. 6-10. (All remaining IDs) | Action: SNIPPET_FIX | Reason: CTR_TOO_LOW_FOR_PAGE_1 | What makes it wrong: Our rule is blind to "Intent". It assumes a low CTR is our fault, but sometimes it's just the nature of the specific keyword.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak Picks: The rule mechanically works, but its biggest weakness is ignoring Search Intent. A low CTR on Page 1 isn't always a bad title; it might be a "Zero-Click Search" (e.g., user searches "Capital of France" and Google shows "Paris" directly on the results page). The user gets the answer without clicking our site. Our rule flags this as an error, which is a false positive.

Leakage Check: CLEAN. The baseline strictly relies on past performance metrics (GSC Impressions, Clicks, Position). We did not use any future performance data, nor did we use labels derived from the target variable.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.